# Polymarket Data Exploration

Explore Polymarket prediction markets using the live API.

**No database setup required** - all data comes directly from Polymarket's public API.

See `API_GUIDE.md` in this folder for full documentation.

---

## 1. Setup

In [ ]:
from cuic_quant.notebook import pm
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 150)
pd.set_option('display.max_rows', 20)

print("Ready! Use pm.fetch_markets() and pm.fetch_orderbook() to get live data.")

## 2. Fetch Live Markets

Get real-time market data directly from Polymarket.

In [ ]:
# Fetch active markets from Polymarket API
df = pm.fetch_markets(limit=50, active=True)

print(f"Fetched {len(df)} markets from Polymarket API")
print()
df[['question', 'yes_price', 'volume', 'status']].head(15)

## 3. Market Analysis Dashboard

Visual overview of market data - probability distribution, volume patterns, and top markets.

In [ ]:
# Sort by volume to find most active markets
top_markets = df.sort_values('volume', ascending=False)

print("TOP MARKETS BY VOLUME")
print("=" * 80)
for i, row in top_markets.head(10).iterrows():
    print(f"${row['volume']:>12,.0f} | {row['yes_price']:.0%} YES | {row['question'][:50]}...")

In [ ]:
# Market Analysis Dashboard
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Polymarket Live Data Dashboard', fontsize=16, fontweight='bold')

# Color palette
colors = ['#6366f1', '#22c55e', '#f59e0b', '#ef4444']

# 1. Price Distribution (Top Left)
ax1 = axes[0, 0]
prices = df['yes_price'].dropna()
n, bins, patches = ax1.hist(prices, bins=20, edgecolor='white', alpha=0.8)
# Color bars by position
for i, patch in enumerate(patches):
    color_val = (bins[i] + bins[i+1]) / 2  # midpoint
    if color_val < 0.3:
        patch.set_facecolor('#ef4444')  # red - unlikely
    elif color_val > 0.7:
        patch.set_facecolor('#22c55e')  # green - likely
    else:
        patch.set_facecolor('#6366f1')  # purple - uncertain
ax1.axvline(0.5, color='#1f2937', linestyle='--', linewidth=2, alpha=0.7)
ax1.set_xlabel('Probability (Yes Price)', fontsize=11)
ax1.set_ylabel('Number of Markets', fontsize=11)
ax1.set_title('Market Probability Distribution', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 1)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# 2. Volume vs Price Scatter (Top Right)
ax2 = axes[0, 1]
volumes = df['volume'].fillna(0)
sizes = np.clip(volumes / volumes.max() * 500, 20, 500)  # scale for bubble size
scatter = ax2.scatter(
    df['yes_price'], 
    volumes,
    s=sizes, 
    c=df['yes_price'], 
    cmap='RdYlGn',
    alpha=0.6,
    edgecolors='white',
    linewidths=0.5
)
ax2.set_xlabel('Probability (Yes Price)', fontsize=11)
ax2.set_ylabel('Volume ($)', fontsize=11)
ax2.set_title('Volume vs Probability', fontsize=12, fontweight='bold')
ax2.set_yscale('log')
ax2.set_xlim(0, 1)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
# Add colorbar
cbar = plt.colorbar(scatter, ax=ax2, shrink=0.8)
cbar.set_label('Probability', fontsize=10)

# 3. Top 10 Markets by Volume (Bottom Left)
ax3 = axes[1, 0]
top10 = df.nlargest(10, 'volume')[['question', 'volume', 'yes_price']].copy()
top10['short_q'] = top10['question'].str[:35] + '...'
top10 = top10.iloc[::-1]  # reverse for horizontal bar
bar_colors = ['#22c55e' if p > 0.5 else '#ef4444' for p in top10['yes_price']]
bars = ax3.barh(top10['short_q'], top10['volume'], color=bar_colors, alpha=0.8, edgecolor='white')
ax3.set_xlabel('Volume ($)', fontsize=11)
ax3.set_title('Top 10 Markets by Volume', fontsize=12, fontweight='bold')
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
# Add probability labels on bars
for bar, prob in zip(bars, top10['yes_price']):
    ax3.text(bar.get_width() + bar.get_width()*0.02, bar.get_y() + bar.get_height()/2,
             f'{prob:.0%}', va='center', fontsize=9, fontweight='bold')

# 4. Market Stats Summary (Bottom Right)
ax4 = axes[1, 1]
ax4.axis('off')

# Calculate stats
total_volume = df['volume'].sum()
avg_price = df['yes_price'].mean()
high_confidence = len(df[(df['yes_price'] > 0.8) | (df['yes_price'] < 0.2)])
uncertain = len(df[(df['yes_price'] > 0.4) & (df['yes_price'] < 0.6)])

stats_text = f"""
MARKET STATISTICS
{'─' * 35}

Total Markets:        {len(df):,}
Total Volume:         ${total_volume:,.0f}
Average Probability:  {avg_price:.1%}

High Confidence:      {high_confidence} markets
  (>80% or <20%)

Uncertain:            {uncertain} markets
  (40-60% range)

Most Active:
  {df.loc[df['volume'].idxmax(), 'question'][:45]}...
  Volume: ${df['volume'].max():,.0f}
"""

ax4.text(0.1, 0.9, stats_text, transform=ax4.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f3f4f6', edgecolor='#d1d5db', alpha=0.8))

plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

## 4. Order Book Data

Fetch the bid/ask order book for a specific market.

Order books show the liquidity available at different price levels.

In [ ]:
import requests
import json

# Fetch open markets with token IDs
response = requests.get(
    "https://gamma-api.polymarket.com/markets", 
    params={"limit": 30, "closed": "false", "active": "true"}
)
raw_markets = response.json()

# Find a market with an orderbook
for market in raw_markets:
    clob_tokens = market.get("clobTokenIds")
    if not clob_tokens:
        continue
    
    if isinstance(clob_tokens, str):
        try:
            clob_tokens = json.loads(clob_tokens)
        except:
            continue
    
    if not clob_tokens:
        continue
    
    token_id = clob_tokens[0]  # YES token
    question = market.get("question", "Unknown")
    
    try:
        orderbook = pm.fetch_orderbook(token_id)
        
        if len(orderbook) > 0:
            print(f"Market: {question[:70]}...")
            print()
            
            bids = orderbook[orderbook['side'] == 'bid'].sort_values('price', ascending=False)
            asks = orderbook[orderbook['side'] == 'ask'].sort_values('price', ascending=True)
            
            print(f"BIDS ({len(bids)} levels)")
            if len(bids) > 0:
                print(bids[['price', 'size']].head(5).to_string())
            
            print(f"\nASKS ({len(asks)} levels)")
            if len(asks) > 0:
                print(asks[['price', 'size']].head(5).to_string())
            
            if len(bids) > 0 and len(asks) > 0:
                spread = asks['price'].min() - bids['price'].max()
                print(f"\nSpread: {spread:.4f} ({spread*100:.2f}%)")
            break
    except:
        continue
else:
    print("No markets with active orderbooks found")

## 5. Sports Markets (NBA)

Sports markets are organized under the `/events` endpoint.

In [ ]:
# Fetch NBA events
response = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"limit": 100, "closed": "false"}
)
events = response.json()

# Filter for NBA
nba_events = [e for e in events if "nba" in e.get("slug", "").lower()]

print(f"Found {len(nba_events)} NBA events")
print()

for event in nba_events[:5]:
    title = event.get("title", "N/A")
    markets = event.get("markets", [])
    total_volume = sum(float(m.get("volume", 0)) for m in markets)
    
    print(f"{title}")
    print(f"  Markets: {len(markets)} | Volume: ${total_volume:,.0f}")
    
    # Show top market by volume
    if markets:
        top = max(markets, key=lambda x: float(x.get("volume", 0)))
        print(f"  Top: {top.get('question', 'N/A')[:60]}...")
    print()

## 6. Direct API Access

For more control, use the Polymarket API directly.

In [ ]:
# Gamma API - Markets endpoint
response = requests.get(
    "https://gamma-api.polymarket.com/markets",
    params={
        "limit": 10,
        "active": "true",
        "closed": "false"
    }
)

markets = response.json()
print(f"Raw API response: {len(markets)} markets")
print()

# Show available fields
if markets:
    print("Available fields:")
    for key in sorted(markets[0].keys()):
        print(f"  - {key}")

---

## Summary

| Method | Description |
|--------|-------------|
| `pm.fetch_markets(limit, active)` | Get live markets from API |
| `pm.fetch_orderbook(token_id)` | Get bid/ask order book |
| Direct `requests.get()` | Full API access |

### API Endpoints

- **Markets:** `https://gamma-api.polymarket.com/markets`
- **Events:** `https://gamma-api.polymarket.com/events`
- **Orderbook:** `https://clob.polymarket.com/book?token_id=...`

See `API_GUIDE.md` for more details.